# Lab 2.2 — Hyperparameter Tuning: GridSearch → RandomSearch → Optuna
### Module 2 | AI/ML Intermediate Workshop — Nutanix Engineering

> **Kernel:** `aiml-venv`

**The engineering question:** Given a fixed compute budget, which tuning strategy finds the best model?

Default XGBoost parameters are chosen to work "okay" on most datasets — not to excel on yours. Hyperparameter tuning is the difference between a demo model and a production model.

**What you will build:**
- A systematic tuning pipeline: brute-force → random → intelligent Bayesian search
- An efficiency comparison: PR-AUC gain per minute of compute for each strategy
- A final production-ready tuned model

> **Instructor Note:** Open with: "GridSearch on 9 XGBoost parameters with 3 values each = 3⁹ = 19,683 combinations × 5 folds = 98,415 model fits. At 0.5s each = 13.7 hours. This is why nobody uses full GridSearch in practice." 

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| scikit-learn | `scikit-learn` |
| xgboost | `xgboost` |
| optuna | `optuna` |
| scipy | `scipy` |
| pandas | `pandas` |
| numpy | `numpy` |
| matplotlib | `matplotlib` |

**Install all at once:**
```bash
pip install scikit-learn xgboost optuna scipy pandas numpy matplotlib
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


## Environment Check

In [ ]:
import subprocess, sys
required = {'xgboost':'xgboost','optuna':'optuna','sklearn':'scikit-learn',
            'pandas':'pandas','numpy':'numpy','matplotlib':'matplotlib','scipy':'scipy'}
for pkg, inst in required.items():
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',inst,'--quiet'])
print("All packages ready ✅")

## Imports

In [ ]:
import warnings, time, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, optuna

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     RandomizedSearchCV, StratifiedKFold,
                                     cross_val_score)
from sklearn.metrics import average_precision_score, f1_score
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Imports OK ✅')

---
## Step 1: Load Data & Establish the Baseline

Load `features_engineered.csv` and recreate the exact train/test split from Lab 2.1.  
Train a default XGBoost — this is the score every tuning strategy must beat.

> **Instructor Note:** Emphasise: the baseline is trained on the **training set only**. The test set is locked away and touched exactly once — at the very end.

In [ ]:
DATA_PATH = os.path.join('..', 'Module_1', 'features_engineered.csv')

try:
    df = pd.read_csv(DATA_PATH)
    DATA_SOURCE = 'Module 1 CSV'
except FileNotFoundError:
    np.random.seed(RANDOM_STATE)
    n = 800
    cpu  = np.random.uniform(5, 100, n)
    resp = np.clip(np.random.exponential(120, n), 1, 700)
    df = pd.DataFrame({
        'cpu_percent'          : cpu,
        'memory_mb'            : np.random.uniform(2048, 32768, n),
        'disk_io_mbps'         : np.clip(np.random.exponential(40, n), 0, 400),
        'response_time_ms'     : resp,
        'hour_of_day'          : np.random.randint(0, 24, n),
        'is_business_hours'    : np.random.randint(0, 2, n),
        'error_rate_per_host'  : np.random.poisson(2, n),
        'cpu_rolling_mean_5'   : cpu + np.random.normal(0, 3, n),
        'memory_rolling_mean_5': np.random.uniform(2048, 32768, n),
        'cpu_memory_ratio'     : cpu / 16,
        'io_per_cpu'           : np.random.exponential(1, n),
        'log_response_time'    : np.log1p(resp),
        'host_encoded'         : np.random.randint(0, 8, n),
        'loglevel_ERROR'       : np.random.randint(0, 2, n),
        'comp_Stargate'        : np.random.randint(0, 2, n),
        'comp_Cerebro'         : np.random.randint(0, 2, n),
        'is_high_cpu'          : (cpu > 80).astype(int),
        'is_slow_response'     : (resp > 500).astype(int),
    })
    DATA_SOURCE = 'Synthetic fallback'

df['is_anomaly'] = df['is_high_cpu'].astype(int)
DROP = [c for c in ['is_high_cpu','is_slow_response','message','date'] if c in df.columns]
df = df.drop(columns=DROP)
feature_cols = [c for c in df.columns if c != 'is_anomaly']
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

X = df[feature_cols]
y = df['is_anomaly']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
# Validation split from training data (used during tuning only)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)

# Baseline
t0 = time.time()
baseline = XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0)
baseline.fit(X_tr, y_tr)
baseline_time = time.time() - t0
baseline_prauc = average_precision_score(y_val, baseline.predict_proba(X_val)[:,1])

print(f'Data source  : {DATA_SOURCE}')
print(f'Train shape  : {X_train.shape}  |  Test: {X_test.shape}')
print(f'Anomaly rate : {y.mean():.1%}')
print(f'\n{"="*45}')
print(f'BASELINE  PR-AUC (val): {baseline_prauc:.4f}')
print(f'BASELINE  Train time  : {baseline_time:.3f}s')
print(f'{"="*45}')
print('\nThis is the score every strategy must beat.')

---
## Step 2: XGBoost Hyperparameter Reference

Before defining search spaces, you need to know what you are searching over.

| Parameter | Controls | Direction to reduce overfitting |
|---|---|---|
| `n_estimators` | Number of trees (100–1000) | Lower (use early stopping) |
| `max_depth` | Max tree depth (3–10) | Lower |
| `learning_rate` | Step size per tree (0.01–0.3) | Lower |
| `subsample` | Row fraction per tree (0.5–1.0) | Lower |
| `colsample_bytree` | Feature fraction per tree (0.5–1.0) | Lower |
| `min_child_weight` | Min sum of instance weights in leaf (1–10) | Higher |
| `gamma` | Min loss reduction to make a split (0–5) | Higher |
| `reg_alpha` | L1 regularisation (0–1) | Higher |
| `reg_lambda` | L2 regularisation (0–10) | Higher |

**Rule of thumb:** `learning_rate` and `n_estimators` are the most impactful pair. Lower `learning_rate` almost always improves accuracy if you compensate with more trees.

---
## Step 3: Strategy 1 — GridSearchCV

GridSearch exhaustively evaluates every combination in the defined grid.  
Use a **deliberately small grid** (only 2 parameters, 2–3 values each) to keep runtime manageable.

**Your task:** Define the grid, run GridSearchCV, and record the best CV score and time.

> **Instructor Note:** After running, show the maths: `len(param_grid_values_combinations) × n_splits` = total model fits. Then project to a full 9-parameter grid to motivate the next strategies.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    'max_depth'     : [3, 5, 7],
    'learning_rate' : [0.05, 0.1, 0.2],
}

grid_model = XGBClassifier(
    n_estimators=100, random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0)

t0 = time.time()
grid_search = GridSearchCV(
    grid_model,
    param_grid,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

grid_prauc = grid_search.best_score_
n_fits = len(grid_search.cv_results_['mean_test_score']) * 3  # × n_splits

print(f'GridSearch results')
print(f'  Grid combinations : {len(grid_search.cv_results_["mean_test_score"])}')
print(f'  Total model fits  : {n_fits}')
print(f'  Total time        : {grid_time:.1f}s')
print(f'  Best params       : {grid_search.best_params_}')
print(f'  Best CV PR-AUC    : {grid_prauc:.4f}')
print(f'  vs baseline       : {grid_prauc - baseline_prauc:+.4f}')

# Heatmap of grid results
scores = grid_search.cv_results_['mean_test_score'].reshape(len(param_grid['max_depth']),
                                                             len(param_grid['learning_rate']))
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pd.DataFrame(scores,
                          index=param_grid['max_depth'],
                          columns=param_grid['learning_rate']),
            annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
ax.set_xlabel('learning_rate'); ax.set_ylabel('max_depth')
ax.set_title('GridSearch PR-AUC Heatmap')
plt.tight_layout()
plt.savefig('gridsearch_heatmap.png', dpi=120)
plt.show()

---
## Step 4: The Combinatorial Explosion Problem

With just 2 parameters and 3 values each, GridSearch ran `3 × 3 × 3 folds = 27 fits`.

Now consider a realistic search across **all 9 XGBoost parameters**:

```
3 values × 9 params = 3⁹ = 19,683 combinations
× 5 folds           = 98,415 model fits
× ~0.5s per fit     = ~13.7 hours
```

**This is not a theoretical problem** — teams actually waste days on naive GridSearch.

**Insight from Bergstra & Bengio (2012):** For most hyperparameter spaces, random search with the same number of trials covers the space *more efficiently* than grid search. Why? Because not all parameters matter equally — random search spends the same number of trials but explores a wider range of the *important* parameters.

---
## Step 5: Strategy 2 — RandomizedSearchCV

Define a **full search space** using continuous distributions (not discrete grids).  
Use `n_iter=40` trials — far fewer than the 98,415 GridSearch would need for the same space.

> **Instructor Note:** The `scipy.stats` distributions are key — `uniform(a, b)` samples from [a, a+b], `randint(a, b)` from [a, b). Using distributions rather than lists means the sampler explores the continuous space properly.

In [ ]:
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators'     : randint(100, 600),
    'max_depth'        : randint(3, 10),
    'learning_rate'    : uniform(0.01, 0.29),
    'subsample'        : uniform(0.5, 0.5),
    'colsample_bytree' : uniform(0.5, 0.5),
    'min_child_weight' : randint(1, 10),
    'reg_alpha'        : uniform(0.0, 1.0),
    'reg_lambda'       : uniform(0.5, 9.5),
}

rand_model = XGBClassifier(
    random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0)

t0 = time.time()
rand_search = RandomizedSearchCV(
    rand_model,
    param_dist,
    n_iter=40,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0,
)
rand_search.fit(X_train, y_train)
rand_time = time.time() - t0

rand_prauc = rand_search.best_score_

print(f'RandomizedSearch results')
print(f'  Iterations        : 40 (vs {n_fits} for GridSearch)')
print(f'  Total time        : {rand_time:.1f}s  (vs {grid_time:.1f}s)')
print(f'  Best params       : {rand_search.best_params_}')
print(f'  Best CV PR-AUC    : {rand_prauc:.4f}')
print(f'  vs baseline       : {rand_prauc - baseline_prauc:+.4f}')
print(f'  vs GridSearch     : {rand_prauc - grid_prauc:+.4f}')

# Score distribution across all trials
fig, ax = plt.subplots(figsize=(8, 4))
trial_scores = rand_search.cv_results_['mean_test_score']
ax.hist(trial_scores, bins=15, color='steelblue', edgecolor='white')
ax.axvline(trial_scores.max(), color='red', linestyle='--', label=f'Best={trial_scores.max():.3f}')
ax.axvline(baseline_prauc, color='grey', linestyle=':', label=f'Baseline={baseline_prauc:.3f}')
ax.set_xlabel('CV PR-AUC'); ax.set_ylabel('Count')
ax.set_title('RandomSearch: Score Distribution Across 40 Trials')
ax.legend()
plt.tight_layout()
plt.savefig('random_search_distribution.png', dpi=120)
plt.show()

---
## Step 6: Strategy 3 — Optuna (Bayesian Optimisation)

Unlike random search, Optuna **learns from previous trials** and focuses sampling on promising regions.  
It uses Tree-structured Parzen Estimators (TPE) by default — a probabilistic model of the objective function.

**Key difference:** After 10 random warmup trials, Optuna's next suggestions are informed by what worked so far. Random search keeps sampling blindly.

> **Instructor Note:** Show the optimisation history plot — the characteristic pattern is: random-looking for the first 10–15 trials, then the scores start trending upward as TPE kicks in. Students should see this pattern clearly.

In [ ]:
def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 600),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'     : RANDOM_STATE,
        'eval_metric'      : 'aucpr',
        'verbosity'        : 0,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                              scoring='average_precision', n_jobs=-1)
    return scores.mean()

t0 = time.time()
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=False)
optuna_time = time.time() - t0

optuna_prauc = study.best_value
best_params  = study.best_params

print(f'Optuna results')
print(f'  Trials            : 50')
print(f'  Total time        : {optuna_time:.1f}s')
print(f'  Best CV PR-AUC    : {optuna_prauc:.4f}')
print(f'  vs baseline       : {optuna_prauc - baseline_prauc:+.4f}')
print(f'  vs RandomSearch   : {optuna_prauc - rand_prauc:+.4f}')
print(f'\nBest parameters:')
for k, v in best_params.items():
    print(f'  {k:<22}: {v}')

In [ ]:
# Visualise Optuna optimisation history
trial_values = [t.value for t in study.trials]
best_so_far  = pd.Series(trial_values).cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(range(len(trial_values)), trial_values, alpha=0.5, s=20, color='steelblue', label='Trial score')
axes[0].plot(range(len(best_so_far)), best_so_far, color='red', lw=2, label='Best so far')
axes[0].axhline(baseline_prauc, color='grey', linestyle='--', label=f'Baseline={baseline_prauc:.3f}')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('CV PR-AUC')
axes[0].set_title('Optuna: Optimisation History'); axes[0].legend(fontsize=9)

# Parameter importance
try:
    importances = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame({'param': list(importances.keys()),
                           'importance': list(importances.values())}).sort_values('importance')
    axes[1].barh(imp_df['param'], imp_df['importance'], color='darkorange', edgecolor='white')
    axes[1].set_xlabel('Relative Importance')
    axes[1].set_title('Optuna: Parameter Importance')
except Exception:
    axes[1].text(0.5, 0.5, 'Parameter importance\nnot available', ha='center', va='center')

plt.tight_layout()
plt.savefig('optuna_history.png', dpi=120)
plt.show()

---
## Step 7: Three-Strategy Comparison

Which strategy gave the best return on compute time?

> **Instructor Note:** Students often expect Optuna to always win by a large margin. On small datasets it may not — because cross-validation noise dominates. The real Optuna advantage appears on larger datasets or longer training runs. Ask: "What would the comparison look like with n_trials=200?".

In [ ]:
summary = pd.DataFrame({
    'Baseline'      : {'Trials/Fits': '—',      'Time (s)': round(baseline_time,1),
                       'Best CV PR-AUC': round(baseline_prauc,4), 'vs Baseline': '—'},
    'GridSearch'    : {'Trials/Fits': n_fits,   'Time (s)': round(grid_time,1),
                       'Best CV PR-AUC': round(grid_prauc,4),
                       'vs Baseline': f'{grid_prauc-baseline_prauc:+.4f}'},
    'RandomSearch'  : {'Trials/Fits': 40,        'Time (s)': round(rand_time,1),
                       'Best CV PR-AUC': round(rand_prauc,4),
                       'vs Baseline': f'{rand_prauc-baseline_prauc:+.4f}'},
    'Optuna'        : {'Trials/Fits': 50,        'Time (s)': round(optuna_time,1),
                       'Best CV PR-AUC': round(optuna_prauc,4),
                       'vs Baseline': f'{optuna_prauc-baseline_prauc:+.4f}'},
}).T

print(summary.to_string())

# Efficiency: PR-AUC gain per minute of compute
strategies = ['GridSearch','RandomSearch','Optuna']
pr_gains    = [grid_prauc-baseline_prauc, rand_prauc-baseline_prauc, optuna_prauc-baseline_prauc]
times_min   = [grid_time/60, rand_time/60, optuna_time/60]
efficiency  = [g/t if t>0 else 0 for g,t in zip(pr_gains, times_min)]

print(f'\nEfficiency (PR-AUC gain / minute):')
for s, e in zip(strategies, efficiency):
    print(f'  {s:<15}: {e:+.4f}')

---
## Step 8: Train Final Tuned Model & Evaluate on Test Set

Retrain the best Optuna parameters on the **full training set** (not just cross-validation folds).  
Then evaluate on the **held-out test set** for the first time.

**Critical:** If the test PR-AUC is much lower than the CV PR-AUC, you have overfit the search process itself (called "tuning overfitting" or "selection bias"). A gap > 0.05 is a warning sign.

> **Instructor Note:** This is a common production mistake — teams report CV scores, not test scores. Always lock the test set away and open it only once.

In [ ]:
final_model = XGBClassifier(
    **best_params,
    random_state=RANDOM_STATE,
    eval_metric='aucpr',
    verbosity=0,
)
final_model.fit(X_train, y_train)

y_prob_test = final_model.predict_proba(X_test)[:, 1]
test_prauc  = average_precision_score(y_test, y_prob_test)
test_f1     = f1_score(y_test, (y_prob_test >= 0.5).astype(int), zero_division=0)

print(f'Final model evaluation on HELD-OUT TEST SET')
print(f'{"="*45}')
print(f'  CV PR-AUC (Optuna best) : {optuna_prauc:.4f}')
print(f'  Test PR-AUC             : {test_prauc:.4f}')
gap = optuna_prauc - test_prauc
print(f'  Gap (CV - Test)         : {gap:+.4f}  {"⚠️  possible overfitting" if gap > 0.05 else "✅ acceptable"}')
print(f'  Test F1                 : {test_f1:.4f}')

# Save
joblib.dump(final_model, 'best_tuned_model.pkl')
results_json = {
    'best_params'   : best_params,
    'cv_prauc'      : round(optuna_prauc,4),
    'test_prauc'    : round(test_prauc,4),
    'test_f1'       : round(test_f1,4),
    'tuning_strategy': 'Optuna TPE, 50 trials',
}
with open('tuning_results.json','w') as f: json.dump(results_json, f, indent=2)
print('\nSaved: best_tuned_model.pkl')
print('Saved: tuning_results.json')

---
## Checkpoint — Discussion Questions

1. **GridSearch with 2 params and 3 values = 27 fits. Why would a 3-param grid with 4 values each be disproportionately more expensive?**
2. **Optuna uses Bayesian optimisation. After 10 warmup trials, how does it decide where to sample next? What does the optimisation history plot tell you about when TPE "kicked in"?**
3. **Your final test PR-AUC may differ from the best CV PR-AUC. What are two reasons this gap could be larger than expected?**
4. **A colleague suggests running Optuna with n_trials=500 to find an even better model. What are the risks of this approach?**

---
## Key Takeaways

| Strategy | Best for | Weakness |
|---|---|---|
| GridSearch | Tiny grids (1–2 params), precise control | Combinatorial explosion |
| RandomSearch | Wide coverage fast, good baseline tuner | Blind — no learning between trials |
| Optuna | Best accuracy for fixed budget, large spaces | Slight overhead per trial; overkill for tiny grids |

**Next:** Lab 2.3 — Cross-Validation & Advanced Evaluation Metrics

---
## 🎯 Assignment — Challenges

### Challenge 1 — Tune LightGBM with Optuna

Write a new Optuna objective function that tunes `LGBMClassifier` instead of XGBoost.  
LightGBM's key parameters differ:

| Parameter | Suggest type | Range |
|---|---|---|
| `num_leaves` | int | 20–300 |
| `max_depth` | int | 3–12 |
| `learning_rate` | float (log) | 0.01–0.3 |
| `n_estimators` | int | 100–600 |
| `min_child_samples` | int | 5–100 |
| `feature_fraction` | float | 0.4–1.0 |
| `bagging_fraction` | float | 0.4–1.0 |
| `reg_alpha` | float (log) | 1e-8–1.0 |

Run 40 trials. Does tuned LightGBM beat tuned XGBoost on test PR-AUC?  
Plot both optimisation histories on the same axes.

In [ ]:
# Challenge 1 — Your solution here



### Challenge 2 — Early Stopping in Optuna

XGBoost supports early stopping: training stops if val score doesn't improve for N rounds.  
This prevents overfitting on `n_estimators` and speeds up each trial.

**Tasks:**
1. Modify the Optuna objective to use early stopping:
   - Pass `eval_set=[(X_val, y_val)]` to `.fit()`
   - Set `early_stopping_rounds=20` in the XGBClassifier constructor
   - Use `trial.suggest_int('n_estimators', 300, 1000)` (higher upper bound — early stopping will cut it)
2. Run 40 trials with early stopping
3. Plot: actual trees used (from `model.best_iteration_`) vs suggested `n_estimators` per trial
4. Compare total tuning time vs Step 6 (no early stopping). How much time did early stopping save?

In [ ]:
# Challenge 2 — Your solution here



### Challenge 3 — Optuna Pruning

Optuna can kill unpromising trials early (pruning) using intermediate validation scores.  
This dramatically speeds up search — imagine stopping a bad trial after 100 trees instead of running all 500.

**Tasks:**
1. Implement a pruning-aware objective using `trial.report()` and `trial.should_prune()`  
   — report the cross-validation score after each fold, prune if below median
2. Create a study with `pruner=optuna.pruners.MedianPruner(n_startup_trials=10)`
3. Run 80 trials with pruning enabled
4. Report: how many trials were pruned? What was the total time vs 50 trials without pruning?
5. Did the best PR-AUC improve with more (but pruned) trials?

*`from optuna.integration import XGBoostPruningCallback` is an alternative approach if you prefer it*

In [ ]:
# Challenge 3 — Your solution here

